# Multi-Temporal Crop Segmentation Example

Purpose: Illustrate crop segmentation.

See: [GitHub Example](https://github.com/NASA-IMPACT/Prithvi-EO-2.0/blob/main/examples/example_multitemporalcrop.ipynb)

In [1]:
import os
import sys
import numpy as np
import torch

import terratorch
from terratorch.datamodules import MultiTemporalCropClassificationDataModule
from terratorch.tasks import SemanticSegmentationTask
from terratorch.datasets.transforms import FlattenTemporalIntoChannels, UnflattenTemporalFromChannels

import albumentations
from albumentations import Compose, Flip
from albumentations.pytorch import ToTensorV2

import lightning.pytorch as pl
from lightning.pytorch.loggers import TensorBoardLogger
from lightning.pytorch.callbacks import ModelCheckpoint

INFO:terratorch:granitewxc not installed, please use pip install granitewxc


## Download Data

In [ ]:
DATASET_PATH = "./data_crop"

In [3]:
# This can a while...
from huggingface_hub import snapshot_download

repo_id = "ibm-nasa-geospatial/multi-temporal-crop-classification"
_ = snapshot_download(repo_id=repo_id, repo_type="dataset", cache_dir="./cache", local_dir=DATASET_PATH)

Fetching 9 files:   0%|          | 0/9 [00:00<?, ?it/s]

validation_data.txt:   0%|          | 0.00/10.0k [00:00<?, ?B/s]

training_data.txt:   0%|          | 0.00/40.1k [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.80k [00:00<?, ?B/s]

chips_df.csv:   0%|          | 0.00/362k [00:00<?, ?B/s]

training_dst.png:   0%|          | 0.00/52.8k [00:00<?, ?B/s]

validation_chips.tgz:   0%|          | 0.00/1.18G [00:00<?, ?B/s]

training_chips.tgz:   0%|          | 0.00/10.6G [00:00<?, ?B/s]

ChunkedEncodingError: ('Connection broken: IncompleteRead(3016622794 bytes read, 7584396132 more expected)', IncompleteRead(3016622794 bytes read, 7584396132 more expected))

In [ ]:
# Adding augmentations for a temporal dataset requires additional transforms
train_transforms = [
    terratorch.datasets.transforms.FlattenTemporalIntoChannels(),
    albumentations.Flip(),
    albumentations.pytorch.transforms.ToTensorV2(),
    terratorch.datasets.transforms.UnflattenTemporalFromChannels(n_timesteps=NUM_FRAMES),
]

# This datamodule allows access to the dataset in its various splits.
data_module = MultiTemporalCropClassificationDataModule(
    data_root=DATASET_PATH,
    train_transform=train_transforms,
    expand_temporal_dimension=True,
)

In [ ]:
# Checking the dataset means and stds
data_module.means, data_module.stds

In [ ]:
# Checking train split size
data_module.setup("fit")
train_dataset = data_module.train_dataset
len(train_dataset)

In [ ]:
# Checking available bands
train_dataset.all_band_names

In [ ]:
# Checking dataset classes
train_dataset.class_names

In [ ]:
# Ploting a few samples
for i in range(5):
    train_dataset.plot(train_dataset[i])

In [ ]:
# Checking validation split size
val_dataset = data_module.val_dataset
len(val_dataset)

In [ ]:
# Checking test split
data_module.setup("test")
test_dataset = data_module.test_dataset
len(test_dataset)

## Configure Model

In [ ]:
OUT_DIR = "./multicrop_example"  # where to save checkpoints and log files

BATCH_SIZE = 16
EPOCHS = 50
LR = 2.0e-4
WEIGHT_DECAY = 0.1
HEAD_DROPOUT=0.1
FREEZE_BACKBONE = False

BANDS = ["BLUE", "GREEN", "RED", "NIR_NARROW", "SWIR_1", "SWIR_2"]
NUM_FRAMES = 3

CLASS_WEIGHTS = [
    0.386375, 0.661126, 0.548184, 0.640482, 0.876862, 0.925186, 3.249462, 
    1.542289, 2.175141, 2.272419, 3.062762, 3.626097, 1.198702
]

SEED = 0